# Chit 1 baseline: reading out a qubit · **FPGA-ns**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a tiny quantized network on raw I/Q readout traces (800 numbers per shot), measure the readout fidelity on the held-out shots, and check that the hls4ml emulation is bit-exact. It copies the published design's shape: 800 → 4 → 2 with **ternary** first-layer weights.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~5 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~2 min: it installs Brevitas and hls4ml) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["brevitas", "hls4ml==1.3.0", "quantizers<2", "qonnx", "onnxoptimizer", "onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
A superconducting qubit stores a 0 or a 1 as one of two energy states: |0⟩ (ground) and |1⟩ (excited). To read it, the electronics send a short microwave pulse
to a resonator coupled to the qubit and record the wave that comes back. The returned wave is slightly different for |0⟩ and |1⟩ (a small shift of its phase).
The FPGA must turn one recording into one decision, within tens of nanoseconds, so that quantum error correction can react before the qubit changes.
**One decision of the model** = "this measurement (one *shot*) found the qubit in |1⟩" (or in |0⟩).

### 0.2 What one sample is
The ADC records the returned wave as two numbers per time step: **I** (in-phase) and **Q** (quadrature), the two components of the wave
measured against a reference wave. One time step is one ADC clock cycle, about **3.25 ns**. The full recording has 770 time steps (2.5 µs); the loader keeps steps 100 to 499.
So one shot = 400 time steps × (I, Q) = **800 numbers**, stored interleaved as [I, Q, I, Q, ...], in raw **ADC counts** (integers).
**Label:** 0 = the qubit was *prepared* in |0⟩; 1 = it was prepared in |1⟩ (by a microwave "π pulse" that flips the qubit, just before the readout). The label is the state that was prepared,
not necessarily the state that was there during the readout: keep that in mind below.

In [ ]:
import matplotlib.pyplot as plt
from capstone.data.qubit import load, START, STOP
NS = 3.25                                         # ns per ADC clock cycle: the 2.5 µs readout = 770 cycles (Di Guglielmo et al. 2025)
X0, y0 = load("val")                              # 100,000 shots: plenty to look at (the model trains on "train")
I0, Q0 = X0[:, 0::2], X0[:, 1::2]                 # undo the interleaving: (shots, 400) each
print(f"one shot = {X0.shape[1]} numbers = {I0.shape[1]} time samples × (I, Q), in ADC counts")
print(f"time samples {START} to {STOP - 1} of the 770 recorded, i.e. {START * NS:.0f} ns to {STOP * NS:.0f} ns after the readout pulse starts")
print("first shot, first three time samples (I, Q, I, Q, I, Q):", X0[0, :6].astype(int), "| its label:", y0[0])
print(f"all values: {X0.min():.0f} to {X0.max():.0f} counts; mean {X0.mean():.0f}, standard deviation {X0.std():.0f}")

In [ ]:
d = X0[y0 == 1].mean(0) - X0[y0 == 0].mean(0)                   # the difference of the two mean traces (800 numbers)
proj = (X0 - X0[y0 == 0].mean(0)) @ d / (d @ d)                  # one number per shot: 0 = looks like the mean |0>, 1 = like the mean |1>
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
bins = np.linspace(-3500, 3500, 141)
for k, c in ((0, "C0"), (1, "C3")):
    ax[0].hist(X0[y0 == k][:2000].ravel(), bins, density=True, histtype="step", color=c, label=f"|{k}⟩ (all 800 values of 2,000 shots)")
    ax[1].hist(proj[y0 == k], np.linspace(-3, 4, 141), histtype="step", color=c, label=f"|{k}⟩")
ax[0].set(xlabel="ADC counts (one I or Q value)", ylabel="density", title="The raw numbers: |0⟩ and |1⟩ look the same")
ax[1].set(xlabel="projection on (mean |1⟩ − mean |0⟩): 0 = mean |0⟩, 1 = mean |1⟩", ylabel="shots", yscale="log",
          title="One number per shot: two peaks that overlap")
for a in ax:
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()
thr = np.linspace(-1, 2, 601)
fid = [0.5 * (np.mean(proj[y0 == 0] <= t) + np.mean(proj[y0 == 1] > t)) for t in thr]
print(f"a threshold on this one number (best threshold {thr[np.argmax(fid)]:.2f}): fidelity {max(fid):.3f} on these shots")
print(f"|1> shots that look like |0> (projection < 0.5): {np.mean(proj[y0 == 1] < 0.5):.1%};  |0> shots that look like |1> (> 0.5): {np.mean(proj[y0 == 0] > 0.5):.1%}")

**Left:** one I or Q value alone tells you almost nothing. Both states give the same wide noise (standard deviation ≈ 840 counts);
the difference between the states is a shift of only about 100 to 250 counts in I and in Q (less at the start of the window), and it only appears when you combine many time steps.
**Right:** projecting each whole shot onto the difference of the two mean traces (one number per shot: 800 multiplications and additions, a "matched filter")
gives two clear peaks. A threshold on this one number already reaches a fidelity of about 0.96, about what the baseline network reaches below.
But about 6% of the |1⟩ shots sit at the |0⟩ peak or between the peaks, and about 2% of the |0⟩ shots sit at the |1⟩ peak. The paper explains why: a qubit in |1⟩ can decay to |0⟩
during the readout (≈ 4% of the shots), and ≈ 1.6% of the qubits are already excited by heat before the measurement. A shot that decayed half-way through the window
lands between the peaks. No model can fix a shot whose label does not match the state that was really there; a better model can only win on the shots between the peaks.

### 0.3 Look at it
Top: the mean I and Q traces of each state over time, with ± 1 standard deviation of the shots as a band. Bottom left: one single shot of each state.
Bottom right: every shot reduced to one point, its I and Q averaged over the window.

In [ ]:
t = np.arange(START, STOP) * NS
fig, ax = plt.subplots(2, 2, figsize=(13, 7.5))
for k, c in ((0, "C0"), (1, "C3")):
    for a, V, name in ((ax[0, 0], I0, "I"), (ax[0, 1], Q0, "Q")):
        m, s = V[y0 == k].mean(0), V[y0 == k].std(0)
        a.fill_between(t, m - s, m + s, color=c, alpha=0.12)
        a.plot(t, m, color=c, label=f"|{k}⟩: mean (line) ± 1 std (band)")
    ax[1, 0].plot(t, I0[y0 == k][0], color=c, lw=0.6, alpha=0.8, label=f"|{k}⟩: one shot")
    ax[1, 1].scatter(I0[y0 == k][:3000].mean(1), Q0[y0 == k][:3000].mean(1), s=2, color=c, alpha=0.4, label=f"|{k}⟩ (3,000 shots)")
for a, name in ((ax[0, 0], "I"), (ax[0, 1], "Q"), (ax[1, 0], "I")):
    a.axvspan(0, START * NS, color="0.85"); a.axvspan(STOP * NS, 770 * NS, color="0.85")
    a.axvline(START * NS, color="k", ls="--", lw=0.8); a.axvline(STOP * NS, color="k", ls="--", lw=0.8)
    a.set(xlim=(0, 770 * NS), xlabel="time after the readout pulse starts [ns]", ylabel=f"{name} [ADC counts]")
    a.legend(fontsize=8, loc="lower right")
ax[0, 0].set(ylim=(-1200, 1200), title="Mean I trace (dashed: the readout window; grey: not in the data)")
ax[0, 1].set(ylim=(-1200, 1200), title="Mean Q trace")
ax[1, 0].set(title="Single shots: the noise hides the difference")
ax[1, 1].set(xlabel="I averaged over the window [ADC counts]", ylabel="Q averaged over the window [ADC counts]", title="Each shot as one point (time-averaged I, Q)")
ax[1, 1].legend(fontsize=8, markerscale=4)
plt.tight_layout(); plt.show()

**What to notice.** The mean traces of |0⟩ and |1⟩ differ by a few hundred counts in both I and Q, far less than the noise band.
In the first few hundred nanoseconds of the window the traces are still settling and the two states are closer together; after that the difference stays flat until the end.
In a single shot the difference cannot be seen by eye. Averaging a shot over its 400 time steps reduces the noise by √400 = 20, and two clouds appear.
The red points inside the blue cloud are |1⟩ shots that look like |0⟩ (decay during the readout, or a wrong preparation).
A shorter window is a faster readout but averages fewer steps, so the clouds overlap more: that is the trade-off behind `load(window=...)`.

### 0.4 The numbers that matter

In [ ]:
_, y_tr0 = load("train", max_items=100_000)       # (labels only; the full training split has 800,000 shots)
_, y_te0 = load("test")
print("split   shots      |0>      |1>")
for name, y, note in (("train", y_tr0, " (first 100,000 of 800,000)"), ("val", y0, ""), ("test", y_te0, "")):
    print(f"{name:<6} {len(y):>7,}   {np.mean(y == 0):6.1%}   {np.mean(y == 1):6.1%}{note}")

**Classes:** balanced (half of the shots were prepared in each state).
**Splits:** the Zenodo record ships a training file (900,000 shots) and a test file (100,000 shots). `test` is the published test file, unchanged.
`val` is 100,000 random shots from the training file (fixed seed), and `train` is the other 800,000. Every shot is an independent measurement,
so a random split does not leak information between splits.
**Value range:** integers from about −4,700 to +4,400 counts. The baseline divides them by one power of two (a bit shift on the FPGA) before its 8-bit input quantizer.

### 0.5 The metric, explained on the data
**Fidelity** F = ½ [P(read 0 | prepared 0) + P(read 1 | prepared 1)]: the mean of the accuracy on |0⟩ shots and the accuracy on |1⟩ shots.
Error correction needs both states read correctly, so both errors count equally, whatever the mix of states in a data set.
Here the classes are balanced, so F is equal to the accuracy. But the two errors are not equal: above, |1⟩ is misread about three times as often as |0⟩,
because |1⟩ can decay during the readout. Print both error rates for your design, not only F.

In [ ]:
del X0, I0, Q0, proj

## 1. Data
One shot = 400 time samples of I and Q (interleaved). Label 0 = |g⟩, 1 = |e⟩. The test split is the published one.

In [ ]:
from capstone.data.qubit import load, fidelity
N_TRAIN = 50_000 if FAST else 300_000             # a subset of the 800k training shots
X_tr, y_tr = load("train", max_items=N_TRAIN)
X_va, y_va = load("val", max_items=20_000 if FAST else None)
X_te, y_te = load("test")
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape, "| fraction |e>:", y_te.mean().round(3))

**Input scaling.** The traces are ADC counts. We divide them by one **power of two** (a bit shift on the FPGA) chosen on the training split, so most values
land in [-1, 1] for the 8-bit input quantizer.

In [ ]:
SCALE = 2.0 ** np.ceil(np.log2(np.quantile(np.abs(X_tr), 0.999)))
Xtr, Xva, Xte = X_tr / SCALE, X_va / SCALE, X_te / SCALE
print("scale:", SCALE)

## 2. The model, costed before training
800 → 4 → 2. First layer **2-bit weights** (with `qat_mlp` that is exactly ternary: −1, 0, +1, so its "multiplications" are additions), second layer 6-bit;
all inputs 8-bit.

In [ ]:
from common.compress import qat_mlp
SIZES = (800, 4, 2)
CFG = [(2, 8), (6, 8)]
model = qat_mlp(CFG, SIZES)
p = profile(model, (800,))
print(f"params {p['params']:,} | MACs {p['macs']:,}")

## 3. Train and measure the fidelity

In [ ]:
fit(model, Xtr, y_tr, epochs=2 if FAST else 8, bs=512, lr=3e-3)
F_TEST = fidelity(y_te, predict(model, Xte).argmax(1))
print(f"readout fidelity (test): {F_TEST:.4f}   (val: {fidelity(y_va, predict(model, Xva).argmax(1)):.4f})")

## 4. On the FPGA: bit-exact emulation and what hls4ml builds
Resources and latency in ns come from **C-synthesis** (instructors, `common/submit.py`). The budget is ≤ 50 ns at II = 1: fully unrolled (RF = 1).

In [ ]:
from common.hls import to_hls, check_bit_exact, dense_summary
hm, _ = to_hls(model, output_dir="hls_prj_qubit", in_features=800)
diff, agree = check_bit_exact(hm, model, Xte[:2000])
print(f"max |Brevitas - hls4ml| on 2,000 shots: {diff:.3g}, same decision on {100 * agree:.2f}%")
DS = dense_summary(hm)
for r in DS:
    print(r)
MULTS = sum(r["mults_needed"] for r in DS)

In [ ]:
ROW = report("1 qubit", (len(Xtr), len(Xva), len(Xte)), (800,), f"fidelity {F_TEST:.4f}", p,
             f"FPGA RF=1: {MULTS:,} multipliers (dense_summary); ns need C-synthesis; bit-exact={diff == 0}",
             "CC BY 4.0 data (Zenodo, confirmed); code repo has no licence (only its recipe reused)",
             extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.